# Nutrition5k paired regression experiment
In Colab select **Runtime → Change runtime type → GPU**. In My Drive, copy the project folders backend/ and tools/ plus the two original checkpoints into NutriVision-AI/models/. Only code cell 1 requires authentication. Configure any alternate paths there, then run all cells. Each run writes a new private Drive folder, including validation selection, all three heads, split IDs, release checkpoint fingerprints, paired predictions, and the metric table. The [official dataset](https://github.com/google-research-datasets/Nutrition5k) provides the overhead RGBs, dish-level labels, and dish ID lists.

In [ ]:
# 1. Mount Drive and configure the run (only interactive cell)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
project = Path('/content/drive/MyDrive/NutriVision-AI')
models = project / 'models'
data = Path('/content/nutrition5k_dataset')  # or path to existing extracted data
download_official_subset = True  # False if data points to an extracted dataset in Drive
validation_ids = None  # Optional group-safe manifest within the training cohort
output = Path('/content/drive/MyDrive') / 'private_nutrition_runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
for rel in ['tools/compare_nutrition.py', 'tools/fetch_nutrition5k.py', 'backend/ai_service.py']:
    assert (project / rel).is_file(), project / rel
for name in ['dinov2_last4_best.pth', 'nutrition_mlp_best.pth']:
    assert (models / name).is_file(), name
os.chdir(project)
sys.path.insert(0, str(project))
print('Private output directory:', output)
os.environ['NUTRIVISION_MODEL_DIR'] = str(models)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm>=1.0.15,<2'], check=True)

In [ ]:
# 2. Prepare the official data (no Cloud credentials)
if download_official_subset:
    subprocess.run([sys.executable, '-u', '-m', 'tools.fetch_nutrition5k', str(data)], check=True)
for subdir in ['metadata', 'dish_ids/splits', 'imagery/realsense_overhead']:
    assert (data / subdir).is_dir(), data / subdir

## Fit, select, then evaluate
The release backbone extracts the same 768 pooled features and Food-101 logits for all methods. Each dish contributes one overhead RGB. Fit and validation use only official train IDs. The default split holds out whole collection days, joining adjacent cross-midnight sessions within an hour, with a fixed seed (502) and a target of 15% validation. All depth training IDs in the public manifests are from one cafe. An original validation manifest can be supplied if it respects the collection groups. The release model may have trained on validation dishes; release validation metrics are descriptive.

All three new heads train from scratch with 128 hidden units, dropout 0.2, standardized log1p targets, Smooth L1, AdamW (0.001, weight decay 0.01), batches of 64, at most 80 epochs and patience 10. Direct predicts kcal and grams. Food-aware also sees predicted Food-101 probabilities. Density predicts kcal/g and grams, then multiplies them. Each candidate checkpoint is saved. Select the candidate with the lowest sum of its validation MAEs relative to release validation MAEs. This selects an experimental challenger even if release is better. Save selection before the single paired test evaluation; no model is automatically deployed. Historical results are context only.

In [ ]:
# 3. Train all three heads and evaluate the locked challenger
cmd = [sys.executable, '-u', '-m', 'tools.compare_nutrition', str(data), str(output)]
if validation_ids:
    cmd += ['--validation-ids', str(validation_ids)]
subprocess.run(cmd, check=True)

In [ ]:
# 4. Read saved metrics and descriptive breakdowns
import json, pandas as pd
report = json.loads((output / 'report.json').read_text())
display(pd.read_csv(output / 'comparison.csv'))
for split in ['validation', 'test']:
    print(split, len(report[f'{split}_ids']), 'dishes')
    display(pd.DataFrame(report[split]).T)
dish = pd.read_csv(output / 'test_predictions.csv')
for name in report['test']:
    dish['absolute_kcal_error'] = (dish[f'{name}_kcal'] - dish.measured_kcal).abs()
    dish['absolute_mass_error'] = (dish[f'{name}_g'] - dish.measured_g).abs()
    summary = dish.groupby('predicted_food101').agg(n=('dish_id','size'), kcal_mae=('absolute_kcal_error','mean'), mass_mae=('absolute_mass_error','mean'))
    print(name, 'by predicted class (descriptive only)')
    display(summary[summary.n >= 5])
print('Private export:', output, 'Validation selection:', report['selected'])

## Private export
All candidate checkpoints and results are already on Drive. Load the selected checkpoint by the name recorded in selection.json. Review comparison.csv for both prediction errors on identical IDs and review individual dishes in test_predictions.csv. Retain the production model unless independent evidence supports replacement. These experimental heads use a different checkpoint layout from the release nutrition MLP.

In [ ]:
# 5. Verify the saved head loads on a training image (no test-set reuse)
import torch
from tools.compare_nutrition import make_head, predict
from backend.ai_service import NutriVisionService
from PIL import Image, ImageOps
from contextlib import nullcontext
if report['selected'] != 'release':
    os.environ['NUTRIVISION_OPEN_VOCAB'] = '0'
    service = NutriVisionService()
    bundle = torch.load(output / (report['selected'] + '.pth'), weights_only=True, map_location=service.device)
    head = make_head(bundle['x_mean'].numel()).to(service.device).eval()
    head.load_state_dict(bundle['weights'], strict=True)
    path = data / 'imagery/realsense_overhead' / report['fit_ids'][0] / 'rgb.png'
    with Image.open(path) as image, torch.inference_mode():
        batch = service.transform(ImageOps.exif_transpose(image).convert('RGB')).unsqueeze(0).to(service.device)
        mixed = service.device.type == 'cuda' and torch.cuda.is_bf16_supported()
        with torch.autocast('cuda', dtype=torch.bfloat16) if mixed else nullcontext():
            tokens = service.food_model.forward_features(batch)
            features = service.food_model.forward_head(tokens, pre_logits=True)
            logits = service.food_model.forward_head(tokens)
        print('Adapter result [kcal, g]', predict(head, bundle, features.float(), logits.float().softmax(1))[0].tolist())